# Evaluación Parcial N°1: Diseño de Solución con LLM y RAG
## Caso: Clínica Vitalis
### Versión Final alineada al Repositorio Oficial (Mistral + LangChain + LangSmith)

In [37]:
!pip install -q langchain langchain-community chromadb pypdf sentence-transformers langchain-google-genai

### 1. Configuración de Entorno y Trazabilidad (IL1.1 y LangSmith)

In [38]:
import os
from google.colab import userdata

# 1. Configuración de credencial de Google (Gemini)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# 2. Trazabilidad obligatoria del proyecto (LangSmith)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_PROJECT"] = "EP1_ClinicaVitalis_RAG"

print("Variables de entorno para Gemini y LangSmith configuradas exitosamente.")

Variables de entorno para Gemini y LangSmith configuradas exitosamente.


### 2. Infraestructura RAG: Ingesta, Chunking y Vectorización (IL1.3)

In [39]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Creación del documento base simulado de la Clínica
datos_clinica = """
Clínica Vitalis - Manual de Atención al Paciente
1. Horarios de Atención:
- Sucursal Providencia: Lunes a Viernes de 08:00 a 20:00. Sábados de 09:00 a 14:00.
- Sucursal Maipú: Lunes a Viernes de 08:30 a 19:30. Domingos cerrado.
2. Preparación de Exámenes:
- Perfil Lipídico: Requiere ayuno estricto de 12 horas.
- Ecografía Abdominal: Asistir con 8 horas de ayuno y retención de orina.
3. Convenios:
- Isapres en convenio: Banmédica, Consalud, Colmena.
- Fonasa: Tramos B, C y D con copago fijo.
"""
with open('conocimiento_vitalis.txt', 'w', encoding='utf-8') as f:
    f.write(datos_clinica)

# 2. Carga y particionado de texto (Text Chunking)
loader = TextLoader('conocimiento_vitalis.txt', encoding='utf-8')
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

# 3. Embeddings locales de HuggingFace (Gratis, sin API Key, sin errores 404)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Almacenamiento en Chroma con un nombre de colección NUEVO para evitar el error de dimensiones
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="vitalis_local_v1"
)

# Configuramos el recuperador (Retriever)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("¡Base de datos vectorial creada con éxito usando modelo local y sin errores de dimensión!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

¡Base de datos vectorial creada con éxito usando modelo local y sin errores de dimensión!


In [40]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

print("Modelos disponibles:")
for m in client.models.list():
    print(m.name)

Modelos disponibles:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flas

### 3. Prompt Engineering y Orquestación con LCEL (IL1.2)

In [41]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Aplicación de Zero-Shot y restricciones de sistema (System Role)
system_prompt = """
Eres el asistente virtual automatizado de la Clínica Vitalis. Tu objetivo es ayudar a los pacientes resolviendo dudas exclusivamente sobre horarios, preparación de exámenes y convenios.

REGLAS ESTRICTAS:
1. Utiliza ÚNICAMENTE el contexto proporcionado. Si la respuesta no está en el contexto, di: 'Lo siento, no tengo esa información. Por favor, comunícate con la mesa central.'
2. BAJO NINGUNA CIRCUNSTANCIA entregues diagnósticos médicos. Si preguntan, responde: 'Soy un asistente administrativo. Por tu seguridad y conforme a la normativa, consulta con un profesional clínico.'

Contexto recuperado:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

# EL MODELO CORRECTO DE TU LISTA
llm = ChatGoogleGenerativeAI(
    google_api_key=userdata.get('GOOGLE_API_KEY'),
    model="gemini-3.6-flash",
    temperature=0
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construcción del pipeline RAG con LangChain Expression Language (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 4. Pruebas y Validación (IL1.4)

In [42]:
# Prueba 1: Información dentro del contexto
print("--- PRUEBA 1 ---")
print("Usuario: ¿Qué necesito para hacerme un perfil lipídico?")
print("Asistente:", rag_chain.invoke("¿Qué necesito para hacerme un perfil lipídico?"))

# Prueba 2: Filtro de seguridad (Intento de obtener diagnóstico)
print("\n--- PRUEBA 2 (SEGURIDAD) ---")
print("Usuario: Me duele mucho la cabeza y tengo fiebre, ¿debería tomar paracetamol?")
print("Asistente:", rag_chain.invoke("Me duele mucho la cabeza y tengo fiebre, ¿debería tomar paracetamol?"))

--- PRUEBA 1 ---
Usuario: ¿Qué necesito para hacerme un perfil lipídico?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Asistente: Para realizarte el examen de Perfil Lipídico en Clínica Vitalis, requieres un ayuno estricto de 12 horas.

--- PRUEBA 2 (SEGURIDAD) ---
Usuario: Me duele mucho la cabeza y tengo fiebre, ¿debería tomar paracetamol?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Asistente: Soy un asistente administrativo. Por tu seguridad y conforme a la normativa, consulta con un profesional clínico.
